<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><strong>Instructor: Waverly Rose Brim</strong></p>

---

<p align="center"><font size="6"><strong>Act II: The Model Blind to Harm</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/model-is-blind.jpg" alt="Illustration of a clinician in profile wearing a headlamp aimed at a Black person whose head is covered by a white cone." width="640" />
  <p><em>Precision is not the same as perspective.</em></p>
</div>

---

## Start here
This notebook answers three questions:
1. Does a model with no race column still depend on race?
2. Does swapping the eGFR equation move the model's AUROC?
3. Why can AUROC stay still while a threshold decision flips?

**Workflow:** predict -> run -> inspect -> interpret.

## Colab setup
Skip this cell if you are running locally.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

## Cold start
Same cohort, same split, whichever notebook you opened first. Earlier
notebooks do not need to be open.

In [ ]:
from data_supply_chain.bootstrap import cold_start

S = cold_start()
print(f"{len(S.df):,} adults · source={S.source} · real={S.is_real}")

### **See the two feature matrices**
A feature matrix is the table a model learns from: one row per person, one
column per feature. These are the first five real patients in the cohort,
shown two ways, with eGFR from the CKD-EPI 2009 equation:

- **With race:** age, sex, a race column (`black`, 1 = Black), and eGFR.
- **Without race:** the same table with the race column dropped. This is
  Model A's matrix below. Model B has the same three columns, but its eGFR
  comes from the race-free CKD-EPI 2021 equation instead.

Compare the eGFR column across the two tables. Under CKD-EPI 2009, race went
into that number before the column was ever dropped.

In [ ]:
from IPython.display import display

rows = S.df.head(5)
with_race = rows[["age", "female", "black", "egfr_2009"]].astype(float)
without_race = with_race.drop(columns="black")

print("WITH race")
display(with_race.round(1))
print("WITHOUT race (Model A's matrix)")
display(without_race.round(1))
print("eGFR column unchanged after dropping race:",
      with_race["egfr_2009"].equals(without_race["egfr_2009"]))

### **Train the models**
We train **two mortality models**. Each one predicts whether a person died
during follow-up. They differ in exactly one thing: which equation produced
the eGFR column.

| | Model A | Model B |
|---|---|---|
| eGFR equation | CKD-EPI 2009 (uses race) | CKD-EPI 2021 (race-free) |
| Features | age, sex, eGFR | age, sex, eGFR |
| Race column | none | none |

Everything else is identical:
1. **Data:** the same people, split the same way every notebook splits them:
   70% of rows to train on, 30% held back to test on.
2. **Model:** logistic regression. Missing values are filled with the training
   median, and each feature is standardized before fitting.
3. **Score:** AUROC on the 30% test rows. It measures how well the model ranks
   people who died above people who survived (0.5 = coin flip, 1.0 = perfect).

This is Strategy C from step 4, the one almost everybody ships: drop the race
column, keep the race-built features.

**Reading the output:** one line per model.
- `global AUROC`: how well it ranks all test patients.
- `worst-by-race`: AUROC for the race group the model does worst on.
- `Delta global AUROC`: how far apart the two models' AUROCs are.

In [ ]:
from data_supply_chain.audit import subgroup_audit
from data_supply_chain.models import fit_audit_model

S = fit_audit_model(S, "egfr_2021")
S09 = fit_audit_model(S, "egfr_2009")

race_tab, race_summ = subgroup_audit(S.audit_sub, S.y_test, S.predictions, "race", n_boot=200)
_, race_summ_09 = subgroup_audit(S09.audit_sub, S09.y_test, S09.predictions, "race", n_boot=200)

for name, state, summ in (
    ("Model A (CKD-EPI 2009)", S09, race_summ_09),
    ("Model B (CKD-EPI 2021)", S, race_summ),
):
    print(f"{name:22s} global AUROC={state.auroc:.3f}  "
          f"worst-by-race({summ['worst_group']})={summ['worst_auroc']:.3f}")

delta = abs(S.auroc - S09.auroc)
print(f"\nDelta global AUROC = {delta:.4f}  ->  the pooled ranking barely moves; the level shift shows up at a cutoff")

## Why AUROC barely moves
AUROC only cares about **order**. Line everyone up from highest predicted risk
to lowest, then ask: how often is a person who died ranked above a person who
survived? The actual risk numbers never matter with this metric, only who sits above whom.

Switching from the 2009 equation to the 2021 equation changes eGFR values, but
it barely changes that order:

- **Within a race group,** the line-up stays almost the same. The 2021
  equation is a full refit, so it changes more than the race term, but in both
  equations higher creatinine means lower eGFR. People who ranked high on
  kidney function under 2009 mostly still rank high under 2021.
- **Across race groups,** the switch matters more. The 2009 equation
  multiplied every Black patient's eGFR by 1.159, and the 2021 equation drops
  that factor. That is a **level shift**: Black patients' eGFR moves down
  relative to everyone else, so some Black and non-Black pairs trade places.
  In this cohort those swaps are few and partly cancel out, so the overall
  AUROC hardly changes.

Two kinds of cutoff are easy to mix up here:
- This model outputs a **predicted risk of death**. AUROC grades how well it
  ranks those risks.
- A clinical cutoff such as metformin's **eGFR 45** is applied to the eGFR
  value itself, not to the model's risk score.

So a patient's eGFR can move from 49 to 44 and cross the line that changes
their metformin prescription, while the model's ranking, and its AUROC,
barely changes.

This is expected, not a bug. To see a level shift, look where levels matter:
at an eGFR cutoff. That is step 6.

In [ ]:
# Demo, take this opportunity to review the output

from IPython.display import Image, SVG, display
from pathlib import Path
import tempfile

from data_supply_chain import figures as _fig

def _show_workshop_fig(png_name, plot_name, *args):
    png = getattr(_fig, png_name, None)
    if png is not None:
        display(Image(data=png(*args), format="png"))
        return
    out = Path(tempfile.mkdtemp()) / "fig.svg"
    getattr(_fig, plot_name)(*args, out)
    display(SVG(filename=str(out)))

_show_workshop_fig(
    "audit_bar_png", "plot_audit_bar",
    race_tab, S.auroc, "Model B (CKD-EPI 2021): looks fine by race",
)

---

## Maintainer cell (safe to skip in workshop)
This writes the findings the website reads, so the numbers on the page stay
synced to this notebook's output.

In [ ]:
from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path

f = Findings.open()
f.record("act2.auroc.egfr_2009", round(S09.auroc, 4))
f.record("act2.auroc.egfr_2021", round(S.auroc, 4))
f.record("act2.auroc.delta", round(delta, 4))
f.record("act2.race.worst_group", race_summ["worst_group"])
f.record("act2.race.worst_auroc", round(race_summ["worst_auroc"], 4))

if findings_path() is None:
    print("Outside a repo checkout - findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")